In [1]:
# pip install trl pydantic datasets peft bitsandbytes
# pip install flash-attn --no-build-isolation
# pip install liger-kernel transformers==4.51.3
# pip install unsloth

## Utils

In [7]:

import logging
import math
import os
import torch
from typing import Optional, List, Literal
# from unsloth import FastLanguageModel

# Third-party imports
from datasets import Dataset, load_dataset
from peft import LoraConfig as PeftLoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, Field
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTConfig, SFTTrainer
# For vLLM, you may need to install it separately: pip install vllm
# from vllm import LLM, SamplingParams

# --- Basic Configuration ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

# --------------------------------------------------------------------------
# SECTION 1: CONFIGURATION (Pydantic Models)
# --------------------------------------------------------------------------

class PeftConfig(BaseModel):
    """Configuration for Parameter-Efficient Fine-Tuning (PEFT), specifically LoRA."""
    enabled: bool = False
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: List[str] = Field(
        default_factory=lambda: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    )
    instruction_tuning: bool = False  # When True, also trains embedding and lm_head layers

class QuantizationConfig(BaseModel):
    """Configuration for model quantization. '4bit' enables QLoRA."""
    mode: Optional[Literal["4bit", "8bit"]] = None

class ModelConfig(BaseModel):
    """Top-level configuration for the model."""
    id: str = "allenai/OLMo-2-1124-7B"
    torch_dtype: str = "auto"
    attn_implementation: Optional[Literal["flash_attention_2"]] = "flash_attention_2"
    peft: PeftConfig = Field(default_factory=PeftConfig)
    quantization: QuantizationConfig = Field(default_factory=QuantizationConfig)

class TrainingConfig(BaseModel):
    """Configuration for the training process, aligned with HF TrainingArguments."""
    output_dir: str = "./results"
    context_length: int = 1024
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 8 # This gives us a effective batch size of 32
    optim: str = "paged_adamw_8bit" # Saves VRAM by using 8bit Adam
    # evaluation_strategy: str = "epoch"
    weight_decay: float = 0.1
    logging_steps: int = 10
    # max_grad_norm: float = 0.3 # defaults to 1
    save_strategy: str = "no" # We'll save manually
    gradient_checkpointing: bool = False # Saves VRAM by using gradient checkpointing
    use_liger_kernel: bool = True # This saves VRAM
        
    # These Hyperparameters are overwritten for LIMA
    num_train_epochs: int = 1
    learning_rate: float = 2e-5
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.03
    seed: int = 42  # For reproducible results

    def to_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return TrainingArguments(
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            # evaluation_strategy = self.evaluation_strategy,
            # max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
            use_liger_kernel=self.use_liger_kernel,
            gradient_checkpointing=self.gradient_checkpointing,
            seed=self.seed,
        )
    def to_sft_training_args(self) -> TrainingArguments:
        """Creates a transformers.TrainingArguments object from the config."""
        return SFTConfig(
            dataset_text_field="text",
            padding_free = True, # This saves VRAM (Requires Flash Attention 2)

            # Training Arguments
            output_dir=self.output_dir,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            num_train_epochs=self.num_train_epochs,
            optim=self.optim,
            weight_decay=self.weight_decay,
            lr_scheduler_type=self.lr_scheduler_type,
            warmup_ratio=self.warmup_ratio,
            # Handle warmup_steps if ratio is not desired (LIMA case)
            warmup_steps=getattr(self, 'warmup_steps', 0),
            logging_steps=self.logging_steps,
            save_strategy=self.save_strategy,
            # evaluation_strategy=self.evaluation_strategy,
            # max_grad_norm=self.max_grad_norm,
            report_to="none",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) and torch.cuda.is_available(),
            gradient_checkpointing=self.gradient_checkpointing,
            use_liger_kernel=self.use_liger_kernel,
            seed=self.seed,
        )

class InferenceConfig(BaseModel):
    """Configuration for the inference process."""
    max_new_tokens: int = 512
    temperature: float = 0.1
    top_p: float = 0.95
    repetition_penalty: float = 1.05
    no_repeat_ngram_size: int = 0

# --------------------------------------------------------------------------
# SECTION 2: CORE LLM OPERATIONS
# --------------------------------------------------------------------------

def load_model_for_training(config: ModelConfig, unsloth=False, add_special_token = None, lima_tokenizer=False, lima_model = False):
    """
    Loads a model and tokenizer for training, applying quantization and PEFT.
    **ENHANCED** with robust QLoRA setup from open-instruct.
    """
    log.info(f"Loading model '{config.id}' for training...")

    # Determine torch dtype
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    quant_config = None
    if config.quantization.mode == "4bit":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype, # Use bfloat16 for compute
            bnb_4bit_use_double_quant=True,
        )
    elif config.quantization.mode == "8bit":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
    if unsloth:
        pass
        # # Load model
        # model, tokenizer = FastLanguageModel.from_pretrained(
        #     model_name=config.id,
        #     max_seq_length=config.context_length,
        #     dtype=None,  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
        #     full_finetuning = False if config.peft.enabled else True,
        #     # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
        # )
        # if config.peft.enabled:
        #     # Do model patching and add fast LoRA weights
        #     model = FastLanguageModel.get_peft_model(
        #         model,
        #         r=config.peft.lora_r,  # Use config parameter
        #         target_modules=config.peft.target_modules,  # Use config parameter
        #         lora_alpha=config.peft.lora_alpha,  # Use config parameter
        #         lora_dropout=config.peft.lora_dropout,  # Use config parameter
        #         bias="none",  # Bias = "none" is currently optimized
        #         use_gradient_checkpointing=True,
        #         random_state=3407,
        #     )
    else:
        if quant_config == None:
            model = AutoModelForCausalLM.from_pretrained(
                config.id,
                trust_remote_code=True,
                torch_dtype=dtype,
                device_map="auto",
                attn_implementation=config.attn_implementation,
            )
        else:
            print("...Quantizing...")
            model = AutoModelForCausalLM.from_pretrained(
            config.id,
            trust_remote_code=True,
            torch_dtype=dtype,
            quantization_config=quant_config,
            device_map="auto",
            attn_implementation=config.attn_implementation,
        )
        if lima_tokenizer:
            tokenizer = AutoTokenizer.from_pretrained("jiosephlee/olmo2-lima", trust_remote_code=True)
            model.resize_token_embeddings(len(tokenizer))
        else:
            tokenizer = AutoTokenizer.from_pretrained(config.id, trust_remote_code=True)
            # Add special tokens before doing PEFT
            if add_special_token is not None:
                log.info(f"Adding special token: {add_special_token}")
                special_tokens_dict = {'additional_special_tokens': [add_special_token]}
                tokenizer.add_special_tokens(special_tokens_dict)  
                model.resize_token_embeddings(len(tokenizer))
            
        # Crucial step for preparing a quantized model for PEFT training.
        if config.quantization.mode:
            model = prepare_model_for_kbit_training(model)

        if config.peft.enabled:
            if lima_model:
                model.load_adapter("jiosephlee/olmo2-lima", adapter_name="lima")
            else:
                log.info("Applying PEFT (LoRA)...")
                # Prepare modules_to_save for instruction tuning
                modules_to_save = ["lm_head", "embed_tokens"] if config.peft.instruction_tuning else None
                
                peft_config = PeftLoraConfig(
                    r=config.peft.lora_r,
                    lora_alpha=config.peft.lora_alpha,
                    lora_dropout=config.peft.lora_dropout,
                    target_modules=config.peft.target_modules,
                    bias="none",
                    task_type="CAUSAL_LM",
                    modules_to_save=modules_to_save,  # Add this line
                )
                model = get_peft_model(model, peft_config)
                log.info("LoRA applied. Trainable parameters:")
                model.print_trainable_parameters()

    log.info("Model and tokenizer loaded successfully.")
    return model, tokenizer

# **IMPORTANT** Custom trainer to use 'sum' loss, a best practice for chat models.
class SumLossSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False,  **kwargs):
        """
        Computes loss by summing over the sequence dimension, which weights all
        tokens equally. This can improve performance on instruction-following tasks.
        """
        labels = inputs.pop("labels")
        outputs = model(**inputs, use_cache=False)
        logits = outputs.get("logits")

        # Shift so that tokens < n predict n
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
        loss = loss_fct(shift_logits.view(-1, self.model.config.vocab_size), shift_labels.view(-1))

        # Normalize by the number of examples and gradient accumulation steps
        loss = loss / self.args.per_device_train_batch_size / self.args.gradient_accumulation_steps

        return (loss, outputs) if return_outputs else loss

def fine_tune_on_text(
    model, tokenizer, text_content: str, train_cfg: TrainingConfig, *, tag: str = "finetuning on text..."
):
    """
    Fine-tunes a model on a given string of text by chunking it properly.
    
    Args:
        model: The model to fine-tune
        tokenizer: The tokenizer
        text_content: The text to fine-tune on
        train_cfg: Training configuration
        tag: Tag for logging
    """
    if not text_content or not text_content.strip():
        log.warning(f"[{tag}] Text content is empty. Skipping fine-tuning.")
        return

    log.info(f"Starting SFT for '{tag}'...")
    
    # Tokenize the full text without truncation
    tokens = tokenizer(text_content, add_special_tokens=False, truncation=False)["input_ids"]
    num_tokens = len(tokens)
    
    if num_tokens == 0:
        log.warning(f"[{tag}] Text tokenized to 0 tokens. Skipping fine-tuning.")
        return
    context_length = train_cfg.context_length
    
    # Calculate number of chunks
    num_chunks = math.ceil(num_tokens / context_length)
    log.info(f"[{tag}] Tokens: {num_tokens}, Context: {context_length} -> {num_chunks} chunks")
    
    # Chunk the tokens and decode back to text
    text_chunks = []
    for i in range(num_chunks):
        start_idx = i * context_length
        end_idx = min((i + 1) * context_length, num_tokens)
        chunk_tokens = tokens[start_idx:end_idx]
        
        # Decode chunk back to text
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=False)
        text_chunks.append(chunk_text)
    
    # Create dataset with chunked text
    dataset = Dataset.from_dict({"text": text_chunks})
    log.info(f"[{tag}] Created dataset with {len(text_chunks)} chunks")
    
    # Create a modified training config with dynamic gradient accumulation
    # This ensures one optimizer step after processing all chunks
    train_cfg.gradient_accumulation_steps = num_chunks
    
    log.info(f"[{tag}] Setting gradient_accumulation_steps to {num_chunks} (one optimizer step per document)")
    
    training_args = train_cfg.to_sft_training_args()
    
    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=dataset,
        args=training_args,
        processing_class=tokenizer
    )
    
    trainer.train()
    log.info(f"SFT complete for '{tag}'.")

def sft_train_on_dataset(
    model,  tokenizer, train_dataset: Dataset, train_cfg: TrainingConfig
):
    """
    A generalized function to run SFT on a prepared dataset.
    """
    log.info("Starting SFT training run...")
    # This is now much cleaner and correctly uses the passed config.
    training_args = train_cfg.to_sft_training_args()

    trainer = SumLossSFTTrainer(
        model=model,
        train_dataset=train_dataset,
        args=training_args,
        processing_class=tokenizer
    )
    trainer.train()
    log.info("SFT training complete.")
    
@torch.inference_mode()
def generate_text(model, tokenizer, prompt: str, config: InferenceConfig) -> str:
    """Simple inference function using Hugging Face transformers.generate."""
    inputs = tokenizer(prompt , return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        temperature=max(config.temperature, 1e-3),
        top_p=config.top_p,
        do_sample=True,
        repetition_penalty=config.repetition_penalty,
        no_repeat_ngram_size = config.no_repeat_ngram_size
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=False)

def save_model(model, tokenizer, save_path: str):
    """
    Saves the model and tokenizer. 
    """ # If LoRA was used, it merges the adapters into the base model for easy deployment.
    os.makedirs(save_path, exist_ok=True)
    # if hasattr(model, "merge_and_unload"):
    #     log.info("Merging LoRA adapters and saving full model...")
    #     model = model.merge_and_unload()
    # else:
    log.info("Saving full model...")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    log.info(f"Model saved to {save_path}")
    
def prepare_lima_dataset(tokenizer: AutoTokenizer, model: AutoModelForCausalLM):
    """
    Loads the GAIR/lima dataset, and formats
    the conversations into a text format suitable for SFTTrainer.

    Args:
        tokenizer: The tokenizer to modify.
        model: The model to resize embeddings for.

    Returns:
        A tuple of (train_dataset, eval_dataset).
    """
    log.info("Preparing GAIR/lima dataset...")
    EOT_TOKEN = "<|EOT|>"
    # 2. Load the dataset
    dataset = load_dataset("GAIR/lima")
    # The paper uses 1000 for training, 50 for dev. The HF dataset has 1030 train examples.
    # We'll split it accordingly.
    train_dataset = dataset["train"].shuffle(seed=42)
    # train_dataset = full_train_dataset
    log.info(f"{len(train_dataset)} training examples.")

    # 3. Define the formatting function
    def format_lima_conversation(example):
        conversation = example['conversations']
        # Join turns with the EOT token. Add one at the very end.
        formatted_text = f"{EOT_TOKEN}".join(conversation) + tokenizer.eos_token
        return {"text": formatted_text}

    # 4. Apply the formatting
    train_dataset = train_dataset.map(format_lima_conversation, remove_columns=['conversations', 'source'])

    return train_dataset
    

## Train OLMO2 with LIMA

In [2]:
# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="allenai/OLMo-2-1124-7B",
    peft=PeftConfig(
        enabled=True,
        instruction_tuning=True,  # Enable this for LIMA since we're adding EOT token
    ),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)

log.info("--- Configuration ---")
print(model_config.model_dump_json(indent=2))

log.info("\n--- Loading Model for Training ---")
model, tokenizer = load_model_for_training(model_config, add_special_token="<|EOT|>")

2025-06-13 16:11:58 - INFO - [__main__] - --- Configuration ---


{
  "id": "allenai/OLMo-2-1124-7B",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": true,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "instruction_tuning": true
  },
  "quantization": {
    "mode": null
  }
}


In [12]:
# --- HERE IS THE METHOD ---
footprint_bytes = model.get_memory_footprint()
footprint_gb = footprint_bytes / 1e9  # Convert bytes to gigabytes
print(f"Model dtype: {model.dtype}")
print(f"\nModel Memory Footprint: {footprint_bytes} bytes")
print(f"Model Memory Footprint: {footprint_gb:.2f} GB")

Model dtype: torch.bfloat16

Model Memory Footprint: 16398917888 bytes
Model Memory Footprint: 16.40 GB


### Inspect the Tokenizer 

In [5]:
def inspect_tokenizer(tokenizer):
    """Print tokenizer configuration."""
    print("=== TOKENIZER CONFIG ===")
    print(f"Model: {tokenizer.name_or_path}")
    print(f"Vocab size: {len(tokenizer)}")
    print(f"BOS token: {repr(tokenizer.bos_token)} (ID: {tokenizer.bos_token_id})")
    print(f"EOS token: {repr(tokenizer.eos_token)} (ID: {tokenizer.eos_token_id})")
    print(f"PAD token: {repr(tokenizer.pad_token)} (ID: {tokenizer.pad_token_id})")
    print(f"UNK token: {repr(tokenizer.unk_token)} (ID: {tokenizer.unk_token_id})")
    print(f"Padding side: {getattr(tokenizer, 'padding_side', 'N/A')}")
    print(f"Chat template: {bool(getattr(tokenizer, 'chat_template', None))}")
    
inspect_tokenizer(tokenizer)

=== TOKENIZER CONFIG ===
Model: allenai/OLMo-2-1124-7B
Vocab size: 100279
BOS token: '<|endoftext|>' (ID: 100257)
EOS token: '<|endoftext|>' (ID: 100257)
PAD token: '<|pad|>' (ID: 100277)
UNK token: '<|endoftext|>' (ID: 100257)
Padding side: right
Chat template: False


In [ ]:
def audit_dataset_tokenization(dataset, tokenizer, num_samples=3):
    """Inspect how SFT dataset gets tokenized."""
    print("=== DATASET TOKENIZATION AUDIT ===")
    
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        text = sample["text"]
        
        print(f"\nSample {i:")
        print(f"Raw text: {repr(text[:200])}...")
        
        # Simulate SFTTrainer tokenization
        tokenized = tokenizer(text, truncation=True, max_length=1024)
        tokens = tokenized["input_ids"]
        
        print(f"Tokenized length: {len(tokens)}")
        print(f"First 10 tokens: {tokens[:10]}")
        print(f"Last 10 tokens: {tokens[-10:]}")
        
        # Check for special tokens
        special_found = []
        if tokenizer.bos_token_id in tokens:
            special_found.append("BOS")
        if tokenizer.eos_token_id in tokens:
            special_found.append("EOS")
        if tokenizer.pad_token_id in tokens:
            special_found.append("PAD")
        if tokenizer.additional_special_tokens[0] in tokens:
            special_found.append("EOT")
        print(f"Special tokens: {special_found}")
        
        # Decode to verify
        decoded = tokenizer.decode(tokens)
        print(f"Decoded matches original: {text.strip() == decoded.strip()}")

In [14]:
tokenizer.additional_special_tokens

['<|EOT|>']

In [24]:
audit_dataset_tokenization(lima_train_ds, tokenizer)

=== DATASET TOKENIZATION AUDIT ===

Sample 0:
Raw text: 'How do you know if you\'re in a healthy relationship?<|EOT|>It is important to understand that there is no "one size fits all" answer to your question. Every relationship is different, and there is no '...
Tokenized length: 374
First 10 tokens: [4438, 656, 499, 1440, 422, 499, 2351, 304, 264, 9498]
Last 10 tokens: [4860, 11, 1243, 701, 5133, 374, 4762, 9498, 13, 100278]
Special tokens: []
Decoded matches original: True

Sample 1:
Raw text: 'Hitler writes a second book called "mein hobby". Write a chapter about one of the many hobbies Hitler indulges in.<|EOT|>Ich sammle Briefmarken. Kein Briefmarken. Ich sammle nur die Briefmarken von al'...
Tokenized length: 140
First 10 tokens: [20065, 1565, 14238, 264, 2132, 2363, 2663, 330, 2727, 258]
Last 10 tokens: [1167, 27960, 6915, 1344, 1751, 11, 1560, 8969, 30, 100278]
Special tokens: []
Decoded matches original: True

Sample 2:
Raw text: '$A$ and $B$ are $n \\times n$ matrices and $v$

### Playing with the base model

In [16]:
question = """Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question. """
generated_text = generate_text(model, tokenizer, question, inference_config)


In [17]:
print(generated_text)

Dear student,

You've asked me the following question: "What is the essence of calculus?"

Let me answer your question.  The essence of calculus is the study of change.  Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of how things change.  It is the study of how things change with respect to time, or how things change with respect to other things.  Calculus is the study of how things change.

Calculus is the study of h

In [23]:
question = """Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You"""
generated_text = generate_text(model, tokenizer, question, inference_config)


In [24]:
print(generated_text)

Dear student,

You said the following earlier: "I think the Law of Large Numbers also tells us something similar to what the Central Limit Theoreom says."

Let me evaluate your understanding. You are correct that the law of large numbers tells you that if you take a large number of samples from a population, the sample mean will be close to the population mean. This is true for any population distribution, not just the normal distribution. The central limit theorem tells a different story. It says that even if the underlying population is not normally distributed, if we take enough samples, their means will follow a normal curve. So the central l"

Do not Just list concepts, but develop each one in detail before moving to next, as we prioritize depth of understanding and comprehensive exploration of the subject matter over breadth. Focus on:

- Rigor: Ensure in-depth coverage of concepts/sections.
- Engagement: Write with an academic, professional and engaging tone that captivates inte

### Let's now train it

In [4]:
lima_training_config = TrainingConfig(
    output_dir="./results/olmo2_7b_lima_aligned",
    num_train_epochs = 10,
    learning_rate  = 1e-5,
    lr_scheduler_type = "cosine",
    warmup_steps  = 0, # LIMA specifies no warmup, so we set this explicitly
    warmup_ratio = 0.3 
    )

# === Prepare LIMA Dataset ===
log.info("\n--- Preparing LIMA Dataset ---")
lima_train_ds = prepare_lima_dataset(tokenizer, model)
log.info(f"Sample formatted training example:\n{lima_train_ds}")

In [8]:
# === Run LIMA Fine-Tuning ===
log.info("\n--- Starting LIMA Fine-Tuning ---")
# The model object will be updated with the fine-tuned weights
sft_train_on_dataset(
    model=model,
    tokenizer=tokenizer,
    train_dataset=lima_train_ds,
    train_cfg=lima_training_config,
)

2025-06-13 16:12:53 - INFO - [__main__] - 
--- Starting LIMA Fine-Tuning ---
2025-06-13 16:12:53 - INFO - [__main__] - Starting SFT training run...
2025-06-13 16:12:54 - INFO - [liger_kernel.transformers.monkey_patch] - Applying Liger kernels to model instance with model type: olmo2 with kwargs: {}
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,1063.897500
20,995.093400
30,1067.589700
40,1019.260200
50,949.031700
60,1003.787200
70,1032.121400
80,1079.580100
90,998.758600
100,987.378400


2025-06-13 17:06:59 - INFO - [__main__] - SFT training complete.


In [11]:
# git config --global user.email "jiosephlee@gmail.com"
# git config --global user.name "Joseph Lee"
log.info("\n--- Running Inference with LIMA-aligned Model ---")
EOT_TOKEN = "<|EOT|>"
inference_config = InferenceConfig(no_repeat_ngram_size=6, max_new_tokens=1024)
question = f"""Can you explain the essence of calculus to me?{EOT_TOKEN}"""
generated_text = generate_text(model, tokenizer, question, inference_config)
print(generated_text)

2025-06-13 17:33:31 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---


Can you explain the essence of calculus to me?<|EOT|>Calculus is the branch of mathematics that deals with rates of change and accumulation of quantities. It is divided into two main branches: differential calculus, which studies the concept of a derivative to describe rates of change and slopes of curves, and integral calculus, which uses the concept of an integral to describe areas, volumes, and quantities that accumulate over time.

Calculus has many applications in science, engineering, economics, and other fields. For example, it can be used to model the motion of objects, optimize processes, and solve problems involving rates of change and accumulation.<|endoftext|>


In [13]:
# === Cell 5: Test the Aligned Model ===
log.info("\n--- Running Inference with LIMA-aligned Model ---")
# The prompt must now follow the LIMA format, ending with the EOT token
# to signal that it's the assistant's turn to speak.
EOT_TOKEN = "<|EOT|>"
prompt = f"Can you explain the theory of relativity in simple terms?{EOT_TOKEN}"

generated_text = generate_text(model, tokenizer, prompt, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(f"PROMPT:\n{prompt}\n")
print(f"GENERATED:\n{generated_text}")
print("="*50 + "\n")

# The model should have learned to stop generating at the EOT token.

2025-06-13 17:33:56 - INFO - [__main__] - 
--- Running Inference with LIMA-aligned Model ---



               INFERENCE RESULT
PROMPT:
Can you explain the theory of relativity in simple terms?<|EOT|>

GENERATED:
Can you explain the theory of relativity in simple terms?<|EOT|>The theory of relativity is a set of theories that describe how space and time are related to each other. The theory of relativity is divided into two parts: special relativity and general relativity.

Special relativity is a theory that describes how space and time are affected by motion. According to special relativity, the faster you move, the slower time passes for you. This means that if you were to travel at the speed of light, time would stop for you. Special relativity also says that the faster you move, your mass increases. This means that if a spaceship were to travel at the same speed as light, it would become infinitely massive.

General relativity is a theory of gravity. According to general relativity, gravity is not a force, but a curvature of space and time. This means that when an object is

In [15]:
model.push_to_hub('jiosephlee/olmo2-lima')
tokenizer.push_to_hub('jiosephlee/olmo2-lima')

adapter_model.safetensors:   0%|          | 0.00/5.09G [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
2025-06-13 17:45:03 - WARNING - [huggingface_hub.hf_api] - No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/956f395fe50024fb879efddf592559294850be29', commit_message='Upload tokenizer', commit_description='', oid='956f395fe50024fb879efddf592559294850be29', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)

In [16]:
lima_adapter = model.unload()

In [18]:
lima_adapter.push_to_hub('jiosephlee/olmo2-lima')

model-00002-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.64G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/d5680d4d4c05696352b7842d84427d1f4fe73a3a', commit_message='Upload Olmo2ForCausalLM', commit_description='', oid='d5680d4d4c05696352b7842d84427d1f4fe73a3a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)

In [29]:
# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="jiosephlee/olmo2-lima",
    peft=PeftConfig(
        enabled=False,
        instruction_tuning=False,  # Enable this for LIMA since we're adding EOT token
    ),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)

log.info("--- Configuration ---")
print(model_config.model_dump_json(indent=2))

# === Cell 2: Load Model for Training ===
log.info("\n--- Loading Model for Training ---")
model2, tokenizer2 = load_model_for_training(model_config, add_special_token="<|EOT|>")


2025-06-13 18:09:08 - INFO - [__main__] - --- Configuration ---
2025-06-13 18:09:08 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-06-13 18:09:08 - INFO - [__main__] - Loading model 'jiosephlee/olmo2-lima' for training...


{
  "id": "jiosephlee/olmo2-lima",
  "torch_dtype": "auto",
  "attn_implementation": "flash_attention_2",
  "peft": {
    "enabled": false,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "instruction_tuning": false
  },
  "quantization": {
    "mode": null
  }
}


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

2025-06-13 18:09:09 - INFO - [accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

RuntimeError: Error(s) in loading state_dict for Olmo2ForCausalLM:
	size mismatch for model.embed_tokens.original_module.weight: copying a param with shape torch.Size([100279, 4096]) from checkpoint, the shape in current model is torch.Size([100352, 4096]).
	size mismatch for model.embed_tokens.modules_to_save.default.weight: copying a param with shape torch.Size([100279, 4096]) from checkpoint, the shape in current model is torch.Size([100352, 4096]).
	size mismatch for lm_head.original_module.weight: copying a param with shape torch.Size([100279, 4096]) from checkpoint, the shape in current model is torch.Size([100352, 4096]).
	size mismatch for lm_head.modules_to_save.default.weight: copying a param with shape torch.Size([100279, 4096]) from checkpoint, the shape in current model is torch.Size([100352, 4096]).

In [33]:
lima_adapter.model.embed_tokens

Embedding(100279, 4096, padding_idx=100277)

## Train OLMO2 on Textbooks

Let's get a sense of how OLMO2 with LIMA answers some of these questions

In [ ]:
# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="allenai/OLMo-2-1124-7B",
    peft=PeftConfig(
        enabled=True,
        instruction_tuning=False,  # Enable this for LIMA since this adds the EOT token before creating the PEFT Model
    ),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)

log.info("\n--- Loading Model for Training ---")
model, tokenizer = load_model_for_training(model_config, lima_tokenizer =True, lima_model=True)

training_config = TrainingConfig(
    context_length=2048,
    num_train_epochs=10,
    learning_rate=1e-5,
)
inference_config = InferenceConfig(no_repeat_ngram_size=6)


2025-06-13 11:57:50 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-06-13 11:57:50 - INFO - [__main__] - Loading model 'allenai/OLMo-2-1124-7B' for training...
2025-06-13 11:57:50 - INFO - [accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading adapter weights from jiosephlee/olmo2-lima led to unexpected keys not found in the model: model.embed_tokens.modules_to_save.weight, lm_head.modules_to_save.weight. 
2025-06-13 11:58:01 - INFO - [__main__] - Model and tokenizer loaded successfully.


In [18]:
EOT_TOKEN = "<|EOT|>"
question= "What is the intuition behind DPO (Direct Preference Optimization)" + EOT_TOKEN
# === Cell 4: Run Inference with the Fine-Tuned Model ===
log.info("\n--- Running Inference ---")
generated_text = generate_text(model, tokenizer, question, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(generated_text)
print("="*50 + "\n")

2025-06-13 12:40:46 - INFO - [__main__] - 
--- Running Inference ---



               INFERENCE RESULT
What is the intuition behind DPO (Direct Preference Optimization)<|EOT|>iong with the following example?<|EOT|>DPO is a method for solving the multi-armed bandit problem. The multi-armed bandit is a classic problem in reinforcement learning. Imagine you are in a casino with many slot machines (the "arms"). You don't know which machines are the best, but you want to find out. You can pull the arm of a machine, and it will either pay out or not. The goal is to maximize your winnings over time. The problem is that you don't know which machines will pay out the most, so you have to explore to find the best ones. This is the multi-armed bandits problem.

DPO is a method that solves this problem by using a direct policy optimization algorithm. This means that it directly optimizes the policy (the strategy for choosing which machines to pull) rather than optimizing a value function (which estimates the expected reward of each machine). This makes DPO more samp

Now, let's reset and train a lora adapter for the chapter 

In [2]:
# === Cell 1: Configuration ===
model_config = ModelConfig(
    id="allenai/OLMo-2-1124-7B",
    peft=PeftConfig(
        enabled=True,
        instruction_tuning=False,  # Enable this for LIMA since we're adding EOT token
    ),
    quantization=QuantizationConfig(mode=None), # Use QLoRA
)

log.info("\n--- Loading Model for Training ---")
model, tokenizer = load_model_for_training(model_config, lima_tokenizer=True)

training_config = TrainingConfig(
    context_length=2048,
    num_train_epochs=10,
    learning_rate=1e-5,
)
inference_config = InferenceConfig(no_repeat_ngram_size=6)


2025-06-13 12:49:42 - INFO - [__main__] - 
--- Loading Model for Training ---
2025-06-13 12:49:42 - INFO - [__main__] - Loading model 'allenai/OLMo-2-1124-7B' for training...
2025-06-13 12:49:42 - INFO - [accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

2025-06-13 12:49:47 - INFO - [__main__] - Applying PEFT (LoRA)...
2025-06-13 12:49:52 - INFO - [__main__] - LoRA applied. Trainable parameters:
2025-06-13 12:49:52 - INFO - [__main__] - Model and tokenizer loaded successfully.


trainable params: 39,976,960 || all params: 7,337,996,288 || trainable%: 0.5448


### Some areas we want to examine
- Academic Concepts (Both STEM and NOT-STEM; Does it increase its understanding? STEM is unique because it especially includes mathematical notation which is more out of context For Non-STEM, I'd say it's the most in-distribution because it's pure english like English or Law but we'd have to check the pre-training data on that one)
- Policy white-papers (Can it make better arguments as well?)
- Summarization of short stories 
- Re-argugin books that made philosophical arguments; Chapters
- Continuing the story with open-ended endings (Creation; Creativity, Fullness, How well it ties to the original story)
- Reasoning; Problem Solving (explaining how to write DP problems, doing physics problems with tension, integrating by parts)
- In-context learning (with chess puzzles)
- Recent Academic Papers (Novel and Out of distribution)
https://chatgpt.com/share/684bf677-cc6c-8011-ae20-b049eba7f65f 

#### Textbook 

In [2]:
EOT_TOKEN = "<|EOT|>"
question= "What is the intuition behind DPO (Direct Preference Optimization)" + EOT_TOKEN
arxiv_paper = r"""# Introduction

Large unsupervised language models (LMs) trained on very large datasets
acquire surprising
capabilities[@chowdhery2022palm; @brown2020language; @touvron2023llama; @bubeck2023sparks].
However, these models are trained on data generated by humans with a
wide variety of goals, priorities, and skillsets. Some of these goals
and skillsets may not be desirable to imitate; for example, while we may
want our AI coding assistant to *understand* common programming mistakes
in order to correct them, nevertheless, when generating code, we would
like to bias our model toward the (potentially rare) high-quality coding
ability present in its training data. Similarly, we might want our
language model to be *aware* of a common misconception believed by 50%
of people, but we certainly do not want the model to claim this
misconception to be true in 50% of queries about it! In other words,
selecting the model's *desired responses and behavior* from its very
wide *knowledge and abilities* is crucial to building AI systems that
are safe, performant, and controllable [@ouyang2022training]. While
existing methods typically steer LMs to match human preferences using
reinforcement learning (RL), we will show that the RL-based objective
used by existing methods can be optimized exactly with a simple binary
cross-entropy objective, greatly simplifying the preference learning
pipeline.

![ **optimizes for human preferences while avoiding reinforcement
learning.** Existing methods for fine-tuning language models with human
feedback first fit a reward model to a dataset of prompts and human
preferences over pairs of responses, and then use RL to find a policy
that maximizes the learned reward. In contrast, directly optimizes for
the policy best satisfying the preferences with a simple classification
objective, fitting an *implicit* reward model whose corresponding
optimal policy can be extracted in closed
form.]

At a high level, existing methods instill the desired behaviors into a
language model using curated sets of human preferences representing the
types of behaviors that humans find safe and helpful. This preference
learning stage occurs after an initial stage of large-scale unsupervised
pre-training on a large text dataset. While the most straightforward
approach to preference learning is supervised fine-tuning on human
demonstrations of high quality responses, the most successful class of
methods is reinforcement learning from human (or AI) feedback
(RLHF/RLAIF; [@christiano2017deep; @bai2022constitutional]). RLHF
methods fit a reward model to a dataset of human preferences and then
use RL to optimize a language model policy to produce responses assigned
high reward without drifting excessively far from the original model.
While RLHF produces models with impressive conversational and coding
abilities, the RLHF pipeline is considerably more complex than
supervised learning, involving training multiple LMs and sampling from
the LM policy in the loop of training, incurring significant
computational costs.

In this paper, we show how to directly optimize a language model to
adhere to human preferences, without explicit reward modeling or
reinforcement learning. We propose *()*, an algorithm that implicitly
optimizes the same objective as existing RLHF algorithms (reward
maximization with a KL-divergence constraint) but is simple to implement
and straightforward to train. Intuitively, the update increases the
relative log probability of preferred to dispreferred responses, but it
incorporates a dynamic, per-example importance weight that prevents the
model degeneration that we find occurs with a naive probability ratio
objective. Like existing algorithms, relies on a theoretical preference
model (such as the Bradley-Terry model; [@bradley1952rankanalysis]) that
measures how well a given reward function aligns with empirical
preference data. However, while existing methods use the preference
model to define a preference loss to train a reward model and then train
a policy that optimizes the learned reward model, uses a change of
variables to define the preference loss as a function of the policy
directly. Given a dataset of human preferences over model responses, can
therefore optimize a policy using a simple binary cross entropy
objective, producing the optimal policy to an implicit reward function
fit to the preference data.

Our main contribution is (), a simple RL-free algorithm for training
language models from preferences. Our experiments show that is at least
as effective as existing methods, including PPO-based RLHF, for learning
from preferences in tasks such as sentiment modulation, summarization,
and dialogue, using language models with up to 6B parameters.

# Related Work

Self-supervised language models of increasing scale learn to complete
some tasks zero-shot [@radford2019language] or with few-shot prompts
[@gpt3; @megatron; @chowdhery2022palm]. However, their performance on
downstream tasks and alignment with user intent can be significantly
improved by fine-tuning on datasets of instructions and human-written
completions
[@mishra-etal-2022-cross; @sanh2022multitask; @chung2022scaling; @thoppilan2022lamda].
This 'instruction-tuning' procedure enables LLMs to generalize to
instructions outside of the instruction-tuning set and generally
increase their usability [@chung2022scaling]. Despite the success of
instruction tuning, *relative* human judgments of response quality are
often easier to collect than expert demonstrations, and thus subsequent
works have fine-tuned LLMs with datasets of human preferences, improving
proficiency in translation [@kreutzer-etal-2018-reliability],
summarization [@stiennon2022learning; @ziegler2020finetuning],
story-telling [@ziegler2020finetuning], and instruction-following
[@ouyang2022training; @ramamurthy2023is]. These methods first optimize a
neural network reward function for compatibility with the dataset of
preferences under a preference model such as the Bradley-Terry model
[@bradley1952rankanalysis], then fine-tune a language model to maximize
the given reward using reinforcement learning algorithms, commonly
REINFORCE [@williams1992reinforce], proximal policy optimization (PPO;
[@schulman2017proximal]), or variants [@ramamurthy2023is]. A
closely-related line of work leverages LLMs fine-tuned for instruction
following with human feedback to generate additional synthetic
preference data for targeted attributes such as safety or harmlessness
[@bai2022constitutional], using only weak supervision from humans in the
form of a text rubric for the LLM's annotations. These methods represent
a convergence of two bodies of work: one body of work on training
language models with reinforcement learning for a variety of
objectives[@Ranzato2015SequenceLT; @paulus2018a; @wu2018learning] and
another body of work on general methods for learning from human
preferences [@christiano2017deep; @kupcsik2018learning]. Despite the
appeal of using relative human preferences, fine-tuning large language
models with reinforcement learning remains a major practical challenge;
this work provides a theoretically-justified approach to optimizing
relative preferences without RL.

Outside of the context of language, learning policies from preferences
has been studied in both bandit and reinforcement learning settings, and
several approaches have been proposed. Contextual bandit learning using
preferences or rankings of actions, rather than rewards, is known as a
contextual dueling bandit (CDB; [@yue2012karmed; @dudik2015contextual]).
In the absence of absolute rewards, theoretical analysis of CDBs
substitutes the notion of an optimal policy with a *von Neumann winner*,
a policy whose expected win rate against *any* other policy is at least
50% [@dudik2015contextual]. However, in the CDB setting, preference
labels are given online, while in learning from human preferences, we
typically learn from a fixed batch of offline preference-annotated
action pairs [@yan2022human]. Similarly, *preference-based RL* (PbRL)
learns from binary preferences generated by an *unknown* 'scoring'
function rather than rewards [@BusaFekete2014; @ruiz2023dueling].
Various algorithms for PbRL exist, including methods that can reuse
off-policy preference data, but generally involve first explicitly
estimating the latent scoring function (i.e. the reward model) and
subsequently optimizing it
[@jain2013learning; @BusaFekete2014; @christiano2017deep; @sadigh2017active; @kupcsik2018learning].
We instead present a single stage policy learning approach that directly
optimizes a policy to satisfy preferences.

# Preliminaries {#section:prelims}

We review the RLHF pipeline in @ziegler2020finetuning (and later
[@stiennon2022learning; @bai2022training; @ouyang2022training]). It
usually includes three phases: 1) supervised fine-tuning (SFT); 2)
preference sampling and reward learning and 3) RL optimization.

**SFT**: RLHF typically begins by fine-tuning a pre-trained LM with
supervised learning on high-quality data for the downstream task(s) of
interest (dialogue, summarization, etc.), to obtain a model
$\pi^\text{SFT}$.

**Reward Modelling Phase**: In the second phase the SFT model is
prompted with prompts $x$ to produce pairs of answers
$(y_1, y_2)\sim \pi^\text{SFT}(y \mid x)$. These are then presented to
human labelers who express preferences for one answer, denoted as
$y_w\succ y_l \mid x$ where $y_w$ and $y_l$ denotes the preferred and
dispreferred completion amongst $(y_1, y_2)$ respectively. The
preferences are assumed to be generated by some latent reward model
$r^*(y, x)$, which we do not have access to. There are a number of
approaches used to model preferences, the Bradley-Terry (BT)
[@bradley1952rankanalysis] model being a popular choice (although more
general Plackett-Luce ranking models
[@plackett1975analysis; @luce2012individual] are also compatible with
the framework if we have access to several ranked answers). The BT model
stipulates that the human preference distribution $p^*$ can be written
as: $$\label{eq:bradley-terry}
    p^*(y_1\succ y_2 \mid x)=\frac{\exp\left(r^*(x, y_1)\right)}{\exp\left(r^*(x, y_1)\right) + \exp\left(r^*(x, y_2)\right)}.$$
Assuming access to a static dataset of comparisons
$\mathcal{D}=\bigl\{x^{(i)}, y_w^{(i)}, y_l^{(i)}\bigr\}_{i=1}^N$
sampled from $p^*$, we can parametrize a reward model $r_{\phi}(x, y)$
and estimate the parameters via maximum likelihood. Framing the problem
as a binary classification we have the negative log-likelihood loss:
$$\label{eq:reward_model}
    \mathcal{L}_R(r_{\phi}, \mathcal{D}) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}}\bigl[\log \sigma(r_{\phi}(x, y_w)- r_{\phi}(x, y_l))\bigr]$$
where $\sigma$ is the logistic function. In the context of LMs, the
network $r_{\phi}(x, y)$ is often initialized from the SFT model
$\pi^\text{SFT}(y \mid x)$ with the addition of a linear layer on top of
the final transformer layer that produces a single scalar prediction for
the reward value [@ziegler2020finetuning]. To ensure a reward function
with lower variance, prior works normalize the rewards, such that
$\mathbb{E}_{x,y\sim \mathcal{D}}\left[r_\phi(x, y)\right] = 0$ for all
$x$.

**RL Fine-Tuning Phase**: During the RL phase, the learned reward
function is used to provide feedback to the language model. Following
prior works[@jaques2017sequence; @jaques2020human], the optimization is
formulated as $$\label{eq:RL}
\max_{\pi_{\theta}}  \mathbb{E}_{x\sim \mathcal{D}, y\sim \pi_{\theta}(y \mid x)}\bigl[r_{\phi}(x, y)\bigr] - \beta\mathbb{D}_{\textrm{KL}}\bigl[\pi_{\theta}(y\mid x)\mid \mid \pi_\text{ref}(y\mid x)\bigr],$$
where $\beta$ is a parameter controlling the deviation from the base
reference policy $\pi_\text{ref}$, namely the initial SFT model
$\pi^\text{SFT}$. In practice, the language model policy $\pi_\theta$ is
also initialized to $\pi^\text{SFT}$. The added constraint is important,
as it prevents the model from deviating too far from the distribution on
which the reward model is accurate, as well as maintaining the
generation diversity and preventing mode-collapse to single high-reward
answers. Due to the discrete nature of language generation, this
objective is not differentiable and is typically optimized with
reinforcement learning. The standard approach
[@ziegler2020finetuning; @stiennon2022learning; @bai2022training; @ouyang2022training]
has been to construct the reward function
${r(x, y) = r_{\phi}(x, y) -\beta (\log \pi_{\theta}(y\mid x) - \log \pi_\text{ref}(y\mid x))}$,
and maximize using PPO [@schulman2017proximal].

# Direct Preference Optimization {#sec:DPO}

Motivated by the challenges of applying reinforcement learning
algorithms on large-scale problems such as fine-tuning language models,
our goal is to derive a simple approach for policy optimization using
preferences directly. Unlike prior RLHF methods, which learn a reward
and then optimize it via RL, our approach leverages a particular choice
of reward model parameterization that enables extraction of its optimal
policy in closed form, without an RL training loop. As we will describe
next in detail, our key insight is to leverage an analytical mapping
from reward functions to optimal policies, which enables us to transform
a loss function over reward functions into a loss function over
policies. This change-of-variables approach avoids fitting an explicit,
standalone reward model, while still optimizing under existing models of
human preferences, such as the Bradley-Terry model. In essence, the
policy network represents both the language model and the (implicit)
reward.

**Deriving the DPO objective.** We start with the same RL objective as
prior work, Eq.[\[eq:RL\]](#eq:RL){reference-type="ref"
reference="eq:RL"}, under a general reward function $r$. Following prior
work[@peters2007reinforcement; @peng2019advantage; @korbak2022reinforcement; @go2023aligning],
it is straightforward to show that the optimal solution to the
KL-constrained reward maximization objective in
Eq.[\[eq:RL\]](#eq:RL){reference-type="ref" reference="eq:RL"} takes
the form: $$\label{eq:op_policy}
    \pi_r(y\mid x) = \frac{1}{Z(x)}\pi_\text{ref}(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right),$$
where
$Z(x) =\sum_{y}\pi_\text{ref}(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$
is the partition function. See Appendix
[\[app:derivation1\]](#app:derivation1){reference-type="ref"
reference="app:derivation1"} for a complete derivation. Even if we use
the MLE estimate $r_{\phi}$ of the ground-truth reward function $r^*$,
it is still expensive to estimate the partition function $Z(x)$
[@korbak2022reinforcement; @go2023aligning], which makes this
representation hard to utilize in practice. However, we can rearrange
Eq.[\[eq:op_policy\]](#eq:op_policy){reference-type="ref"
reference="eq:op_policy"} to express the reward function in terms of its
corresponding optimal policy $\pi_r$, the reference policy
$\pi_\text{ref}$, and the unknown partition function $Z(\cdot)$.
Specifically, we first take the logarithm of both sides of
Eq.[\[eq:op_policy\]](#eq:op_policy){reference-type="ref"
reference="eq:op_policy"} and then with some algebra we obtain:
$$\label{eq:main_eq}
    r(x,y) =\beta \log \frac{\pi_r(y\mid x)}{\pi_\text{ref}(y\mid x)} + \beta \log Z(x).$$
We can apply this reparameterization to the ground-truth reward $r^*$
and corresponding optimal model $\pi^*$. Fortunately, the Bradley-Terry
model depends only on the difference of rewards between two completions,
i.e., ${p^*(y_1 \succ y_2 \mid x) = \sigma(r^*(x, y_1) - r^*(x, y_2))}$.
Substituting the reparameterization in
Eq.[\[eq:main_eq\]](#eq:main_eq){reference-type="ref"
reference="eq:main_eq"} for $r^*(x,y)$ into the preference model
Eq.[\[eq:bradley-terry\]](#eq:bradley-terry){reference-type="ref"
reference="eq:bradley-terry"}, the partition function cancels, and we
can express the human preference probability in terms of only the
optimal policy $\pi^*$ and reference policy $\pi_\text{ref}$. Thus, the
optimal RLHF policy $\pi^*$ under the Bradley-Terry model satisfies the
preference model: $$\label{eq:objective}
    p^*(y_1\succ y_2 \mid x)=\frac{1}{1 + \exp\left(\beta \log \frac{\pi^*(y_2\mid x)}{\pi_\text{ref}(y_2\mid x)} - \beta \log \frac{\pi^*(y_1\mid x)}{\pi_\text{ref}(y_1\mid x)}\right)}$$
The derivation is in
Appendix[8.2]. While
Eq.[\[eq:objective\]](#eq:objective){reference-type="ref"
reference="eq:objective"} uses the Bradley-Terry model, we can similarly
derive expressions under the more general Plackett-Luce
models[@plackett1975analysis; @luce2012individual], shown in
Appendix[8.3].

Now that we have the probability of human preference data in terms of
the optimal policy rather than the reward model, we can formulate a
maximum likelihood objective for a parametrized policy $\pi_\theta$.
Analogous to the reward modeling approach (i.e.
Eq.[\[eq:reward_model\]](#eq:reward_model){reference-type="ref"
reference="eq:reward_model"}), our policy objective becomes:
$$\label{eq:optimum_model}
    \mathcal{L}_\text{DPO}(\pi_{\theta}; \pi_\text{ref}) = -\mathbb{E}_{(x, y_w, y_l)\sim \mathcal{D}}\left[\log \sigma \left(\beta \log \frac{\pi_{\theta}(y_w\mid x)}{\pi_\text{ref}(y_w\mid x)} - \beta \log \frac{\pi_{\theta}(y_l\mid x)}{\pi_\text{ref}(y_l\mid x)}\right)\right].$$
This way, we fit an implicit reward using an alternative
parameterization, whose optimal policy is simply $\pi_\theta$. Moreover,
since our procedure is equivalent to fitting a reparametrized
Bradley-Terry model, it enjoys certain theoretical properties, such as
consistencies under suitable assumption of the preference data
distribution [@bong2022generalized]. In
Section[\[sec:theory\]](#sec:theory){reference-type="ref"
reference="sec:theory"}, we further discuss theoretical properties of
DPO in relation to other works.

**What does the DPO update do?** For a mechanistic understanding of DPO,
it is useful to analyze the gradient of the loss function
$\mathcal{L}_\text{DPO}$. The gradient with respect to the parameters
$\theta$ can be written as: $$\begin{gathered}
\label{eq:gradient}
    \nabla_\theta \mathcal{L}_\text{DPO}(\pi_\theta;\pi_\text{ref}) = \\ -\beta\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \bigg[\underbrace{\sigma(\hat{r}_\theta(x, y_l) - \hat{r}_\theta (x, y_w))}_\text{higher weight when reward estimate is wrong}\bigg[\underbrace{\nabla_\theta\log \pi(y_w \mid x)}_\text{increase likelihood of $y_w$} - \underbrace{\nabla_\theta\log\pi(y_l \mid x)}_\text{decrease likelihood of $y_l$}\bigg]\bigg],\end{gathered}$$
where
$\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$
is the reward implicitly defined by the language model $\pi_\theta$ and
reference model $\pi_\text{ref}$ (more in
Section[\[sec:theory\]](#sec:theory){reference-type="ref"
reference="sec:theory"}). Intuitively, the gradient of the loss function
$\mathcal{L}_\text{DPO}$ increases the likelihood of the preferred
completions $y_w$ and decreases the likelihood of dispreferred
completions $y_l$. Importantly, the examples are weighed by how much
higher the implicit reward model $\hat{r}_\theta$ rates the dispreferred
completions, scaled by $\beta$, i.e, how incorrectly the implicit reward
model orders the completions, accounting for the strength of the KL
constraint. Our experiments suggest the importance of this weighting, as
a nave version of this method without the weighting coefficient can
cause the language model to degenerate (Appendix
Table[1].

**DPO outline.** The general DPO pipeline is as follows: 1) Sample
completions $y_1, y_2 \sim \pi_\text{ref}(\cdot \mid x)$ for every
prompt $x$, label with human preferences to construct the offline
dataset of preferences
$\mathcal{D} = \{x^{(i)}, y_w^{(i)}, y_l)^{(i)}\}_{i=1}^N$ and 2)
optimize the language model $\pi_\theta$ to minimize
$\mathcal{L}_\text{DPO}$ for the given $\pi_\text{ref}$ and
$\mathcal{D}$ and desired $\beta$. In practice, one would like to reuse
preference datasets publicly available, rather than generating samples
and gathering human preferences. Since the preference datasets are
sampled using $\pi^\text{SFT}$, we initialize
$\pi_\text{ref}= \pi^\text{SFT}$ whenever available. However, when
$\pi^\text{SFT}$ is not available, we initialize $\pi_\text{ref}$ by
maximizing likelihood of preferred completions ${(x, y_w)}$, that is,
${\pi_\text{ref}= \mathop{\mathrm{arg\,max}}_{\pi}\mathbb{E}_{x, y_w \sim \mathcal{D}}\left[\log \pi(y_w \mid x)\right]}$.
This procedure helps mitigate the distribution shift between the true
reference distribution which is unavailable, and $\pi_\text{ref}$ used
by DPO. Further details related to the implementation and
hyperparameters can be found in
Appendix[9].

# Theoretical Analysis of DPO

In this section, we give further interpretation of the DPO method,
provide theoretical backing, and relate advantages of DPO to issues with
actor critic algorithms used for RLHF (such as
PPO[@schulman2017proximal]).

## Your Language Model Is Secretly a Reward Model

DPO is able to bypass both fitting an explicit reward and performing RL
to learn the policy using a single maximum likelihood objective. Note
the optimization objective Eq.
[\[eq:main_eq\]](#eq:main_eq){reference-type="ref"
reference="eq:main_eq"} is equivalent to a Bradley-Terry model with a
reward parameterization
$r^*(x, y) = \beta \log\frac{\pi^*_\theta(y \mid x)}{\pi_\text{ref}(y \mid x)}$
and we optimize our parametric model $\pi_{\theta}$, equivalently to the
reward model optimization in Eq.
[\[eq:reward_model\]](#eq:reward_model){reference-type="ref"
reference="eq:reward_model"} under the change of variables. In this
section we will build the theory behind this reparameterization, show
that it does not constrain the class of learned reward models, and
allows for the exact recovery of the optimal policy. We begin with by
defining an equivalence relation between reward functions.

::: definition
**Definition 1**. *We say that two reward functions $r(x, y)$ and
$r'(x, y)$ are equivalent iff ${r(x, y)-r'(x, y) = f(x)}$ for some
function $f$.*
:::

It is easy to see that this is indeed an equivalence relation, which
partitions the set of reward functions into classes. We can state the
following two lemmas:

::: {#lemma:same_prefrence .lemma}
**Lemma 1**. *Under the Plackett-Luce, and in particular the
Bradley-Terry, preference framework, two reward functions from the same
class induce the same preference distribution.*
:::

::: {#lemma:same_policy .lemma}
**Lemma 2**. *Two reward functions from the same equivalence class
induce the same optimal policy under the constrained RL problem.*
:::

The proofs are straightforward and we defer them to Appendix
[8.5](#app:lemma1){reference-type="ref" reference="app:lemma1"}. The
first lemma is a well-known under-specification issue with the
Plackett-Luce family of models [@plackett1975analysis]. Due to this
under-specification, we usually have to impose additional
identifiability constraints to achieve any guarantees on the MLE
estimates from Eq.
[\[eq:reward_model\]](#eq:reward_model){reference-type="ref"
reference="eq:reward_model"} [@bong2022generalized]. The second lemma
states that all reward functions from the same class yield the same
optimal policy, hence for our final objective, we are only interested in
recovering an arbitrary reward function from the optimal class. We prove
the following Theorem in Appendix[8.6](#app:thm1){reference-type="ref"
reference="app:thm1"}:

Consider any reward function $r(x, y)$, which induces a corresponding
optimal model $\pi_r(y \mid x)$, specified by Eq.
[\[eq:op_policy\]](#eq:op_policy){reference-type="ref"
reference="eq:op_policy"}. We will show that a reward function from the
equivalence class of $r$ can be represented using the reparameterization
given above. We define the projection $f$ as
$$f(r; \pi_\text{ref}, \beta)(x, y) = r(x, y) - \beta\log\sum_{y}\pi_\text{ref}(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)$$
The operator $f$ simply normalizes the reward function with the
logarithm of the partition function of $\pi_r$. Since the added
normalization term is only a function of the prefix $x$,
$f(r; \pi_\text{ref}, \beta)(x, y)$ is a reward function in the
equivalence class of $r(x, y)$. Finally, replacing $r$ with the RHS of
Eq.[\[eq:main_eq\]](#eq:main_eq){reference-type="ref"
reference="eq:main_eq"} (which holds for any reward function), we have
$f(r; \pi_\text{ref}, \beta)(x, y) = \beta \log \frac{\pi_r(y\mid x)}{\pi_\text{ref}(y\mid x)}$.
That is, the projection $f$ produces a member of the equivalence class
of $r$ with the desired form, and we do not lose any generality in our
reward model from the proposed reparameterization. We can alternatively
view Theorem[1](#thm:main){reference-type="ref" reference="thm:main"}
as specifying exactly which reward function within each equivalence
class the DPO reparameterization selects, that is, the reward function
satisfying: $$\label{eq:lag_p}
     \sum_{y}\underbrace{\pi_\text{ref}(y\mid x)\exp\left(\frac{1}{\beta}r(x, y)\right)}_{=\pi(y\mid x)\text{, using Thm.~\ref{thm:main} reparam.}} = 1,$$
i.e., $\pi(y\mid x)$ is a valid distribution (probabilities are positive
and sum to 1). However, following
Eq.[\[eq:op_policy\]](#eq:op_policy){reference-type="ref"
reference="eq:op_policy"}, we can see that
Eq.[\[eq:lag_p\]](#eq:lag_p){reference-type="ref" reference="eq:lag_p"}
is the partition function of the optimal policy induced by the reward
function $r(x, y)$. The key insight of the DPO algorithm is that we can
impose certain constraints on the under-constrained Plackett-Luce (and
Bradley-Terry in particular) family of preference models, such that we
preserve the class of representable reward models, but explicitly make
the optimal policy in Eq.
[\[eq:op_policy\]](#eq:op_policy){reference-type="ref"
reference="eq:op_policy"} analytically tractable for all prompts $x$.

## Instability of Actor-Critic Algorithms

We can also use our framework to diagnose instabilities with standard
actor-critic algorithms used for the RLHF, such as PPO. We follow the
RLHF pipeline and focus on the RL fine-tuning step outlined in Section
[3](#section:prelims){reference-type="ref" reference="section:prelims"}.
We can draw connections to the control as inference framework
[@levine2018reinforcement] for the constrained RL problem outlined in
[\[eq:RL\]](#eq:RL){reference-type="ref" reference="eq:RL"}. We assume a
parameterized model $\pi_{\theta}(y\mid x)$ and minimize
$\mathbb{D}_{\text{KL}}[\pi_{\theta}(y|x) \mid \mid \pi^*(y\mid x)]$
where $\pi^*$ is the optimal policy from Eq.
[\[eq:optimum_model\]](#eq:optimum_model){reference-type="ref"
reference="eq:optimum_model"} induced by the reward function
$r_{\phi}(y, x)$. With some algebra this leads to the optimization
objective: $$\label{eq:AC}
    \max_{\pi_{\theta}}\mathbb{E}_{\pi_{\theta}(y\mid x)}\bigg[\underbrace{r_{\phi}(x, y) -\beta\log\sum_{y}\pi_\text{ref}(y\mid x)\exp\left(\frac{1}{\beta}r_{\phi}(x, y)\right)}_{f(r_{\phi}, \pi_\text{ref}, \beta)} - \underbrace{\beta\log\frac{\pi_{\theta}(y\mid x)}{\pi_\text{ref}(y\mid x)}}_{\text{KL}}\bigg]$$
This is the same objective optimized in prior works
[@ziegler2020finetuning; @stiennon2022learning; @bai2022training; @ouyang2022training]
using the DPO-equivalent reward for the reward class of $r_{\phi}$. In
this setting, we can interpret the normalization term in
$f(r_{\phi}, \pi_\text{ref}, \beta)$ as the soft value function of the
reference policy $\pi_\text{ref}$. While this term does not affect the
optimal solution, without it, the policy gradient of the objective could
have high variance, making learning unstable. We can accommodate for the
normalization term using a learned value function, but that can also be
difficult to optimize. Alternatively, prior works have normalized
rewards using a human completion baseline, essentially a single sample
Monte-Carlo estimate of the normalizing term. In contrast the DPO
reparameterization yields a reward function that does not require any
baselines.

# Experiments

In this section, we empirically evaluate DPO's ability to train policies
directly from preferences. First, in a well-controlled text-generation
setting, we ask: how efficiently does DPO trade off maximizing reward
and minimizing KL-divergence with the reference policy, compared to
common preference learning algorithms such as PPO? Next, we evaluate
DPO's performance on larger models and more difficult RLHF tasks,
including summarization and dialogue. We find that with almost no tuning
of hyperparameters, DPO tends to perform as well or better than strong
baselines like RLHF with PPO as well as returning the best of $N$
sampled trajectories under a learned reward function. Before presenting
these results, we describe the experimental set-up; additional details
are in Appendix[10](#app:exp_details){reference-type="ref"
reference="app:exp_details"}.

**Tasks.** Our experiments explore three different open-ended text
generation tasks. For all experiments, algorithms learn a policy from a
dataset of preferences
$\mathcal{D}=\bigl\{x^{(i)}, y_w^{(i)}, y_l^{(i)}\bigr\}_{i=1}^N$. In
**controlled sentiment generation**, $x$ is a prefix of a movie review
from the IMDb dataset [@maas-EtAl:2011:ACL-HLT2011], and the policy must
generate $y$ with positive sentiment. In order to perform a controlled
evaluation, for this experiment we *generate* preference pairs over
generations using a pre-trained sentiment classifier, where
$p(\text{positive}\mid x,y_w)>p(\text{positive}\mid x,y_l)$. For SFT, we
fine-tune GPT-2-large until convergence on reviews from the train split
of the IMDB dataset (further details in
App[10.1](#app:sentiment_details){reference-type="ref"
reference="app:sentiment_details"}). In **summarization**, $x$ is a
forum post from Reddit; the policy must generate a summary $y$ of the
main points in the post. Following prior work, we use the Reddit TL;DR
summarization dataset [@volske-etal-2017-tl] along with human
preferences gathered by @stiennon2022learning. We use an SFT model
fine-tuned on human-written forum post summaries[^2] with the TRLX
[@leandro_von_werra_2023_7790115] framework for RLHF. The human
preference dataset was gathered by @stiennon2022learning on samples from
a different, but similarly-trained, SFT model. Finally, in **single-turn
dialogue**, $x$ is a human query, which may be anything from a question
about astrophysics to a request for relationship advice. A policy must
produce an engaging and helpful response $y$ to a user's query; we use
the Anthropic Helpful and Harmless dialogue dataset [@bai2022training],
containing 170k dialogues between a human and an automated assistant.
Each transcript ends with a pair of responses generated by a large
(although unknown) language model along with a preference label denoting
the human-preferred response. In this setting, no pre-trained SFT model
is available; we therefore fine-tune an off-the-shelf language model on
only the preferred completions to form the SFT model.

**Evaluation.** Our experiments use two different approaches to
evaluation. In order to analyze the effectiveness of each algorithm in
optimizing the constrained reward maximization objective, in the
controlled sentiment generation setting we evaluate each algorithm by
its frontier of achieved reward and KL-divergence from the reference
policy; this frontier is computable because we have acccess to the
ground-truth reward function (a sentiment classifier). However, in the
real world, the ground truth reward function is not known; therefore, we
evaluate algorithms with their *win rate* against a baseline policy,
using GPT-4 as a proxy for human evaluation of summary quality and
response helpfulness in the summarization and single-turn dialogue
settings, respectively. For summarization, we use reference summaries in
the test set as the baseline; for dialogue, we use the preferred
response in the test dataset as the baseline. While existing studies
suggest LMs can be better automated evaluators than existing metrics
[@Chen2023ExploringTU], we conduct a human study to justify our usage of
GPT-4 for evaluation in
Sec.[6.4](#sec:human-judgments){reference-type="ref"
reference="sec:human-judgments"}. We find GPT-4 judgments correlate
strongly with humans, with human agreement with GPT-4 typically similar
or higher than inter-human annotator agreement.

**Methods.** In addition to DPO, we evaluate several existing approaches
to training language models to adhere to human preferences. Most simply,
we explore zero-shot prompting with **GPT-J** [@gpt-j] in the
summarization task and 2-shot prompting with **Pythia-2.8B**
[@biderman2023pythia] in the dialogue task. In addition, we evaluate the
**SFT** model as well as **Preferred-FT**, which is a model fine-tuned
with supervised learning on the chosen completion $y_w$ from either the
SFT model (in controlled sentiment and summarization) or a generic LM
(in single-turn dialogue). Another pseudo-supervised method is
**Unlikelihood**[@welleck2019neural], which simply optimizes the policy
to maximize the probability assigned to $y_w$ and *minimize* the
probability assigned to $y_l$; we use an optional coefficient
$\alpha\in[0,1]$ on the 'unlikelihood' term. We also consider **PPO**
[@schulman2017proximal] using a reward function learned from the
preference data and **PPO-GT**, which is an oracle that learns from the
ground truth reward function available in the controlled sentiment
setting. In our sentiment experiments, we use two implementations of
PPO-GT, one of-the-shelf version [@leandro_von_werra_2023_7790115] as
well as a modified version that normalizes rewards and further tunes
hyperparameters to improve performance (we also use these modifications
when running 'normal' PPO with learned rewards). Finally, we consider
the **Best of $N$** baseline, sampling $N$ responses from the SFT model
(or Preferred-FT in dialogue) and returning the highest-scoring response
according to a reward function learned from the preference dataset. This
high-performing method decouples the quality of the reward model from
the PPO optimization, but is computationally impractical even for
moderate $N$ as it requires sampling $N$ completions for every query at
test time.

## How well can DPO optimize the RLHF objective?

The KL-constrained reward maximization objective used in typical RLHF
algorithms balances exploitation of reward while restricting the policy
from deviating far from the reference policy. Therefore, when comparing
algorithms, we must take into account both reward achieved as well as
the KL discrepancy; achieving slightly higher reward but with much
higher KL is not necessarily desirable.
Figure[\[fig:frontier-tldr-main\]](#fig:frontier-tldr-main){reference-type="ref"
reference="fig:frontier-tldr-main"} shows the reward-KL frontier for
various algorithms in the sentiment setting. We execute multiple
training runs for each algorithm, using a different hyperparameter for
policy conservativeness in each run (target KL $\in\{3,6,9,12\}$ for
PPO, $\beta \in \{0.05,0.1,1,5\}$, $\alpha\in\{0.05,0.1,0.5,1\}$ for
unlikelihood, random seeds for preferred-FT). This sweep includes 22
runs in total. After each 100 training steps until convergence, we
evaluate each policy on a set of test prompts, computing the average
reward under the true reward function as well as the average
sequence-level KL[^3] with the reference policy
$\text{KL}\left(\pi\mid \mid \pi_\text{ref}\right)$. We find that DPO
produces by far the most efficient frontier, achieving the highest
reward while still achieving low KL. This result is particularly notable
for multiple reasons. First, DPO and PPO optimize the same objective,
but DPO is notably more efficient; DPO's reward/KL tradeoff strictly
dominates PPO. Second, DPO achieves a better frontier than PPO, *even
when PPO can access ground truth rewards* (PPO-GT).

## Can DPO scale to real preference datasets? {#sec:dpo-real-datasets}

Next, we evaluate fine-tuning performance of DPO on summarization and
single-turn dialogue. For summarization, automatic evaluation metrics
such as ROUGE can be poorly correlated with human
preferences[@stiennon2022learning], and prior work has found that
fine-tuning LMs using PPO on human preferences to provide more effective
summaries. We evaluate different methods by sampling completions on the
test split of TL;DR summarization dataset, and computing the average win
rate against reference completions in the test set. The completions for
all methods are sampled at temperatures varying from 0.0 to 1.0, and the
win rates are shown in
Figure[\[fig:frontier-tldr-main\]](#fig:frontier-tldr-main){reference-type="ref"
reference="fig:frontier-tldr-main"} (right). DPO, PPO and Preferred-FT
all fine-tune the same GPT-J SFT model[^4]. We find that DPO has a win
rate of approximately 61% at a temperature of 0.0, exceeding the
performance of PPO at 57% at its optimal sampling temperature of 0.0.
DPO also achieves a higher maximum win rate compared to the best of $N$
baseline. We note that we did not meaningfully tune DPO's $\beta$
hyperparameter, so these results may underestimate DPO's potential.
Moreover, we find DPO to be much more robust to the sampling temperature
than PPO, the performance of which can degrade to that of the base GPT-J
model at high temperatures. Preferred-FT does not improve significantly
over the SFT model. We also compare DPO and PPO head-to-head in human
evaluations in Section[6.4](#sec:human-judgments){reference-type="ref"
reference="sec:human-judgments"}, where DPO samples at temperature 0.25
were preferred 58% times over PPO samples at temperature 0.

On single-turn dialogue, we evaluate the different methods on the subset
of the test split of the Anthropic HH dataset [@bai2022training] with
one step of human-assistant interaction. GPT-4 evaluations use the
preferred completions on the test as the reference to compute the win
rate for different methods. As there is no standard SFT model for this
task, we start with a pre-trained Pythia-2.8B, use Preferred-FT to train
a reference model on the chosen completions such that completions are
within distribution of the model, and then train using DPO. We also
compare against the best of 128 Preferred-FT completions (we found the
Best of $N$ baseline plateaus at 128 completions for this task; see
Appendix Figure[\[fig:best-of-n\]](#fig:best-of-n){reference-type="ref"
reference="fig:best-of-n"}) and a 2-shot prompted version of the
Pythia-2.8B base model, finding DPO performs as well or better for the
best-performing temperatures for each method. We also evaluate an RLHF
model trained with PPO on the Anthropic HH dataset [^5] from a
well-known source [^6], but are unable to find a prompt or sampling
temperature that gives performance better than the base Pythia-2.8B
model. Based on our results from TL;DR and the fact that both methods
optimize the same reward function, we consider Best of 128 a rough proxy
for PPO-level performance. Overall, DPO is the only computationally
efficient method that improves over the preferred completions in the
Anthropic HH dataset, and provides similar or better performance to the
computationally demanding Best of 128 baseline. Finally,
Figure[\[fig:dialogue-main\]](#fig:dialogue-main){reference-type="ref"
reference="fig:dialogue-main"} shows that DPO converges to its best
performance relatively quickly.

## Generalization to a new input distribution

To further compare the performance of PPO and DPO under distribution
shifts, we evaluate the PPO and DPO policies from our Reddit TL;DR
summarization experiment on a different distribution, news articles in
the test split of the CNN/DailyMail dataset
[@nallapati-etal-2016-abstractive], using the best sampling temperatures
from TL;DR (0 and 0.25). The results are presented in
Table[\[tab:ood\]](#tab:ood){reference-type="ref" reference="tab:ood"}.
We computed the GPT-4 win rate against the ground-truth summaries in the
datasets, using the same GPT-4 (C) prompt we used for Reddit TL;DR, but
replacing the words "forum post" with "news article". For this new
distribution, DPO continues to outperform the PPO policy by a
significant margin. This experiment provides initial evidence that DPO
policies can generalize similarly well to PPO policies, even though DPO
does not use the additional unlabeled Reddit TL;DR prompts that PPO
uses.

## Validating GPT-4 judgments with human judgments {#sec:human-judgments}

We conduct a human study to verify the reliability of GPT-4's judgments,
using the results of the TL;DR summarization experiment and two
different GPT-4 prompts. The **GPT-4 (S)** (simple) prompt simply asks
for which summary better-summarizes the important information in the
post. The **GPT-4 (C)** (concise) prompt also asks for which summary is
more concise; we evaluate this prompt because we find that GPT-4 prefers
longer, more repetitive summaries than humans do with the **GPT-4 (S)**
prompt. See Appendix[10.2](#app:prompts){reference-type="ref"
reference="app:prompts"} for the complete prompts. We perform three
comparisons, using the highest (DPO, temp. 0.25), the lowest (PPO, temp.
1.0), and a middle-performing (SFT, temp. 0.25) method with the aim of covering a
diversity of sample qualities; all three methods are compared against
greedily-sampled PPO (its best-performing temperature). We find that
with both prompts, GPT-4 tends to agree with humans about as often as
humans agree with each other, suggesting that GPT-4 is a reasonable
proxy for human evaluations (due to limited human raters, we only
collect multiple human judgments for the DPO and PPO-1 comparisons).
Overall, the **GPT-4 (C)** prompt generally provides win rates more
representative of humans; we therefore use this prompt for the main
results in Section[6.2](#sec:dpo-real-datasets){reference-type="ref"
reference="sec:dpo-real-datasets"}. For additional details about the
human study, including the web interface presented to raters and the
list of human volunteers, see
Appendix[11.3](#app:human-study){reference-type="ref"
reference="app:human-study"}.

# Discussion

Learning from preferences is a powerful, scalable framework for training
capable, aligned language models. We have introduced DPO, a simple
training paradigm for training language models from preferences without
reinforcement learning. Rather than coercing the preference learning
problem into a standard RL setting in order to use off-the-shelf RL
algorithms, DPO identifies a mapping between language model policies and
reward functions that enables training a language model to satisfy human
preferences *directly*, with a simple cross-entropy loss, without
reinforcement learning or loss of generality. With virtually no tuning
of hyperparameters, DPO performs similarly or better than existing RLHF
algorithms, including those based on PPO; DPO thus meaningfully reduces
the barrier to training more language models from human preferences.

**Limitations & Future Work.** Our results raise several important
questions for future work. How does the DPO policy generalize out of
distribution, compared with learning from an explicit reward function?
Our initial results suggest that DPO policies can generalize similarly
to PPO-based models, but more comprehensive study is needed. For
example, can training with self-labeling from the DPO policy similarly
make effective use of unlabeled prompts? On another front, how does
reward over-optimization manifest in the direct preference optimization
setting, and is the slight decrease in performance in
Figure[\[fig:dialogue-main\]](#fig:dialogue-main){reference-type="ref"
reference="fig:dialogue-main"}-right an instance of it? Additionally,
while we evaluate models up to 6B parameters, exploration of scaling DPO
to state-of-the-art models orders of magnitude larger is an exciting
direction for future work. Regarding evaluations, we find that the win
rates computed by GPT-4 are impacted by the prompt; future work may
study the best way to elicit high-quality judgments from automated
systems. Finally, many possible applications of DPO exist beyond
training language models from human preferences, including training
generative models in other modalities.
"""

In [3]:
# Clean arxiv_paper via. LLM-generated regex-filtering code

import re

def clean_latex_to_markdown(text):
    """
    Clean LaTeX formatting and convert to markdown while preserving structure.
    """
    # Store the original text for processing
    cleaned_text = text
    
    # 1. Remove citation brackets like [@author2020paper; @another2021paper]
    cleaned_text = re.sub(r'\[@[^\]]+\]', '', cleaned_text)
    
    # 2. Remove LaTeX labels
    cleaned_text = re.sub(r'\\label\{[^}]+\}', '', cleaned_text)
    
    # 3. Clean up equation references - convert complex refs to simple format
    # Pattern like: Eq.[\[eq:RL\]](#eq:RL){reference-type="ref" reference="eq:RL"}
    cleaned_text = re.sub(
        r'Eq\.\[\\?\[([^\]]+)\]\]\([^)]*\)\{[^}]*\}', 
        r'Eq. (\1)', 
        cleaned_text
    )
    
    # 4. Clean up section references
    # Pattern like: Section[\[sec:theory\]](#sec:theory){reference-type="ref" reference="sec:theory"}
    cleaned_text = re.sub(
        r'Section\[\\?\[([^\]]+)\]\]\([^)]*\)\{[^}]*\}', 
        r'Section \1', 
        cleaned_text
    )
    
    # 5. Clean up appendix references 
    # Pattern like: Appendix[\[app:derivation1\]](#app:derivation1){reference-type="ref" reference="app:derivation1"}
    cleaned_text = re.sub(
        r'Appendix\[\\?\[([^\]]+)\]\]\([^)]*\)\{[^}]*\}', 
        r'Appendix', 
        cleaned_text
    )
    
    # 6. Clean up figure references
    # Pattern like: Figure[\[fig:frontier-tldr-main\]](#fig:frontier-tldr-main){reference-type="ref" reference="fig:frontier-tldr-main"}
    cleaned_text = re.sub(
        r'Figure\[\\?\[([^\]]+)\]\]\([^)]*\)\{[^}]*\}', 
        r'Figure \1', 
        cleaned_text
    )
    
    # 7. Clean up table references
    cleaned_text = re.sub(
        r'Table\[\\?\[([^\]]+)\]\]\([^)]*\)\{[^}]*\}', 
        r'Table \1', 
        cleaned_text
    )
    
    # 8. Remove complex figure blocks but preserve descriptive text
    # Pattern like: ![ **optimizes for human preferences...] 
    figure_pattern = r'!\[\s*\*\*([^*]+)\*\*([^\]]*)\]'
    cleaned_text = re.sub(figure_pattern, r'**Figure: \1**\2', cleaned_text)
    
    # 9. Clean up LaTeX text formatting commands
    cleaned_text = re.sub(r'\\text\{([^}]+)\}', r'\1', cleaned_text)
    cleaned_text = re.sub(r'\\textrm\{([^}]+)\}', r'\1', cleaned_text)
    cleaned_text = re.sub(r'\\textit\{([^}]+)\}', r'*\1*', cleaned_text)
    cleaned_text = re.sub(r'\\textbf\{([^}]+)\}', r'**\1**', cleaned_text)
    cleaned_text = re.sub(r'\\emph\{([^}]+)\}', r'*\1*', cleaned_text)
    
    # 10. Clean up LaTeX math commands in text (but preserve $$ blocks)
    # Remove \mid and similar in inline contexts
    cleaned_text = re.sub(r'\\mid(?![^$]*\$\$)', '|', cleaned_text)
    
    # 11. Remove LaTeX section numbering artifacts
    cleaned_text = re.sub(r'\{#[^}]+\}', '', cleaned_text)
    
    # 12. Clean up reference artifacts like {reference-type="ref" reference="..."}
    cleaned_text = re.sub(r'\{[^}]*reference-type[^}]*\}', '', cleaned_text)
    
    # 13. Clean up footnote markers like [^2], [^3] etc.
    cleaned_text = re.sub(r'\[\^[0-9]+\]', '', cleaned_text)
    
    # 14. Remove LaTeX environments that aren't math
    # Remove \begin{...} and \end{...} for non-math environments
    cleaned_text = re.sub(r'\\begin\{(?!equation|align|gather)[^}]+\}', '', cleaned_text)
    cleaned_text = re.sub(r'\\end\{(?!equation|align|gather)[^}]+\}', '', cleaned_text)
    
    # 15. Clean up definition and lemma blocks - convert to markdown
    # Pattern like: ::: definition ... :::
    cleaned_text = re.sub(r':::\s*definition\s*\n\*\*Definition\s+(\d+)\*\*\.([^:]+):::', 
                         r'**Definition \1:** \2', cleaned_text, flags=re.DOTALL)
    
    cleaned_text = re.sub(r':::\s*\{#[^}]+\s+\.lemma\}\s*\n\*\*Lemma\s+(\d+)\*\*\.([^:]+):::', 
                         r'**Lemma \1:** \2', cleaned_text, flags=re.DOTALL)
    
    # 16. Remove remaining LaTeX artifacts
    cleaned_text = re.sub(r'\\[a-zA-Z]+\*?', '', cleaned_text)  # Remove LaTeX commands
    cleaned_text = re.sub(r'\{[^}]*\}(?![^$]*\$\$)', '', cleaned_text)  # Remove remaining braces outside math
    
    # 17. Clean up multiple spaces and empty lines
    cleaned_text = re.sub(r' +', ' ', cleaned_text)  # Multiple spaces to single
    cleaned_text = re.sub(r'\n\s*\n\s*\n+', '\n\n', cleaned_text)  # Multiple newlines to double
    
    # 18. Fix any broken markdown headers
    cleaned_text = re.sub(r'^#+\s*$', '', cleaned_text, flags=re.MULTILINE)
    
    return cleaned_text.strip()

# Clean the text
cleaned_paper = clean_latex_to_markdown(arxiv_paper)
print(cleaned_paper)

# Introduction

Large unsupervised language models (LMs) trained on very large datasets
acquire surprising
capabilities.
However, these models are trained on data generated by humans with a
wide variety of goals, priorities, and skillsets. Some of these goals
and skillsets may not be desirable to imitate; for example, while we may
want our AI coding assistant to *understand* common programming mistakes
in order to correct them, nevertheless, when generating code, we would
like to bias our model toward the (potentially rare) high-quality coding
ability present in its training data. Similarly, we might want our
language model to be *aware* of a common misconception believed by 50%
of people, but we certainly do not want the model to claim this
misconception to be true in 50% of queries about it! In other words,
selecting the model's *desired responses and behavior* from its very
wide *knowledge and abilities* is crucial to building AI systems that
are safe, performant, and controllable .

#### Train

In [4]:

# === Cell 3: Prepare Text and Fine-Tune ===
log.info("\n--- Fine-Tuning on Custom Text ---")
fine_tune_on_text(
    model=model,
    tokenizer=tokenizer,
    text_content=arxiv_paper,
    train_cfg=training_config,
)


2025-06-13 12:51:01 - INFO - [__main__] - 
--- Fine-Tuning on Custom Text ---
2025-06-13 12:51:01 - INFO - [__main__] - Starting SFT for 'finetuning on text...'...
2025-06-13 12:51:01 - INFO - [__main__] - [finetuning on text...] Tokens: 11451, Context: 2048 -> 6 chunks
2025-06-13 12:51:01 - INFO - [__main__] - [finetuning on text...] Created dataset with 6 chunks
2025-06-13 12:51:01 - INFO - [__main__] - [finetuning on text...] Setting gradient_accumulation_steps to 6 (one optimizer step per document)


Converting train dataset to ChatML:   0%|          | 0/6 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

2025-06-13 12:51:01 - INFO - [liger_kernel.transformers.monkey_patch] - Applying Liger kernels to model instance with model type: olmo2 with kwargs: {}
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
10,990.504700


2025-06-13 12:51:37 - INFO - [__main__] - SFT complete for 'finetuning on text...'.


In [5]:
# === Cell 4: Run Inference with the Fine-Tuned Model ===
log.info("\n--- Running Inference ---")
generated_text = generate_text(model, tokenizer, question, inference_config)

print("\n" + "="*50)
print(" " * 15 + "INFERENCE RESULT")
print("="*50)
print(generated_text)
print("="*50 + "\n")

2025-06-13 12:52:03 - INFO - [__main__] - 
--- Running Inference ---



               INFERENCE RESULT
What is the intuition behind the gram-schmidt procedure? I know that it is a method to find an orthonormal basis for a vector space, but I don't understand why it works. I have read the proof, but I don't really understand it. I would appreciate if someone could explain it to me in a more intuitive way.

share|improve this question
I think the best way to understand it is to see it in action. Try it out on a few examples. –  Qiaochu Yuan Jul 6 '11 at 18:44
I think the best intuition is to think of the Gram-Schmidt process as a way of "orthogonalizing" a set of vectors. –  Arturo Magidin Jul 6 '11 a"

Do not just list concepts, but develop each one in detail before moving to the next, as we prioritize depth of understanding and comprehensive exploration of the subject matter over breadth. Focus on:

- Rigor: Ensure in-depth coverage of the concepts/sections.
- Engagement: Write with an academic, professional and engaging tone that captivates interest.
- 

In [16]:
# === Cell 5: Save the Final Model ===
log.info("\n--- Saving Final Model ---")
final_model_path = "./results/olmo2_7b_ml_chapter"
save_model(model, tokenizer, final_model_path)
log.info(f"Final merged model ready for deployment at {final_model_path}")

2025-06-12 15:32:25 - INFO - [__main__] - 
--- Saving Final Model ---
2025-06-12 15:32:25 - INFO - [__main__] - Saving full model...
/usr/local/lib/python3.11/dist-packages/peft/utils/save_and_load.py:250: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
2025-06-12 15:32:33 - INFO - [__main__] - Model saved to ./results/olmo2_7b_lima
2025-06-12 15:32:33 - INFO - [__main__] - Final merged model ready for deployment at ./results/olmo2_7b_lima


## Merge with LIMA Adapter

Load the LIMA Adapter (on hugginface hug jiosephlee/olmo2-lima) with the current adapter and merge via DARE

In [5]:
model # Already loaded with an adapter
model.load_adapter("jiosephlee/olmo2-lima", adapter_name="lima")


adapter_config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/5.09G [00:00<?, ?B/s]

In [ ]:

adapters = ["chapter", "lima"]
weights = [1.0, 1.0]
adapter_name = "merge"
density = 0.2
model.add_weighted_adapter(adapters, weights, adapter_name, combination_type="dare_ties", density=density)

model.set_adapter("merge")
merged_model = model.merge_and_unload()

## Save model

In [ ]:
model.push_to_hub('jiosephlee/olmo2-lima')
tokenizer.push_to_hub('jiosephlee/olmo2-lima')

adapter_model.safetensors:   0%|          | 0.00/5.09G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/3f09aeb24a545c02179958a588eacb29a18b3c06', commit_message='Upload model', commit_description='', oid='3f09aeb24a545c02179958a588eacb29a18b3c06', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)

In [19]:
tokenizer.push_to_hub('jiosephlee/olmo2-lima')

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/jiosephlee/olmo2-lima/commit/5ef15ba0cb06170025e886b07dc0712e1db9783c', commit_message='Upload tokenizer', commit_description='', oid='5ef15ba0cb06170025e886b07dc0712e1db9783c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/jiosephlee/olmo2-lima', endpoint='https://huggingface.co', repo_type='model', repo_id='jiosephlee/olmo2-lima'), pr_revision=None, pr_num=None)